# JSON RAG

Flatten nested JSON into path-annotated facts so structure is not lost.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *

data = json.loads((DATA / "company.json").read_text())

Walk the tree and emit one fact per leaf. The key *path* (`products > Carrier-1 > payload_kg`) preserves the hierarchy.

In [ ]:
def flatten(node, path=()):
    if isinstance(node, dict):
        for key, value in node.items():
            yield from flatten(value, path + (key,))
    elif isinstance(node, list):
        for i, item in enumerate(node):
            label = item.get("name", i) if isinstance(item, dict) else i
            yield from flatten(item, path + (str(label),))
    else:
        yield f"{' > '.join(path)}: {node}"

facts = list(flatten(data))
print(len(facts), "facts")
print("\n".join(facts[:8]))

In [ ]:
store = build_store(facts)
for q in ["How heavy a load can Carrier-1 carry?", "How long is the warranty?", "Who is the CTO?"]:
    print("Q:", q, "\nA:", rag(q, store, k=4)[0], "\n")